# KoMaP 2026 — v2: 8방향 TTA + 앙상블 (Colab)

베이스라인(`komap_unet_colab.ipynb`, ResNet34 U-Net, val mIoU 76.31%)에서 이어서 진행한 실험.
1. 기존 `best.pt`(ResNet34)에 **회전까지 포함한 8방향 TTA**
2. 두 번째 모델 **EfficientNet-B3 U-Net**을 train 70장으로만 학습
3. 단독/앙상블을 valid로 비교 → 가장 좋은 조합으로 test 예측

**대회 규정**: train으로만 학습한 모델로 valid 점수를 내고, 같은 모델로 test를 예측해야 함 → valid는 학습에 쓰지 않음.

**실행**: 런타임 → 모두 실행 (GPU 런타임, Drive의 `KoMaP/checkpoints/best.pt` 필요).
결과는 `KoMaP/predictions_v2/`에 저장하고 기존 `predictions/`는 그대로 둠.

## 0. 준비 (Drive 연결 · 설치 · 데이터)

In [1]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/KoMaP"   # zip 3개를 올린 폴더
DATA_DIR  = "/content/data"                   # 압축 해제 위치 (Colab 로컬 디스크라 빠름)

!pip -q install segmentation-models-pytorch albumentations
!apt-get -qq install -y fonts-nanum > /dev/null
!mkdir -p {DATA_DIR}
# zip을 올렸으면 압축 해제, 폴더째 올렸으면 Drive 아래 어디든 <split>/images 를 찾아 복사
# (셸 for문의 $z는 IPython 변수 치환과 충돌해서 파이썬으로 처리)
import os, shutil, glob, zipfile
for split in ["train", "valid", "test"]:
    dst = f"{DATA_DIR}/{split}"
    z = f"{DRIVE_DIR}/{split}.zip"
    if os.path.exists(z) and not os.path.isdir(f"{dst}/images"):
        zipfile.ZipFile(z).extractall(DATA_DIR)
    if not os.path.isdir(f"{dst}/images"):
        hits = glob.glob(f"{DRIVE_DIR}/**/{split}/images", recursive=True)
        if not hits:
            print(f"❌ {split}/images 를 못 찾음"); continue
        shutil.copytree(os.path.dirname(hits[0]), dst)
    print(f"✅ {split}: 이미지 {len(glob.glob(f'{dst}/images/*.png'))}장, 마스크 {len(glob.glob(f'{dst}/masks/*.png'))}장")
!nvidia-smi --query-gpu=name,memory.total --format=csv

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 6.5 MB/s eta 0:00:00
✅ train: 이미지 70장, 마스크 70장
✅ valid: 이미지 20장, 마스크 20장
✅ test: 이미지 10장, 마스크 0장
name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
import os
import random, time, glob
from pathlib import Path
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import torch, torch.nn as nn
import segmentation_models_pytorch as smp
import albumentations as A
from matplotlib import font_manager
font_manager.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
plt.rcParams["font.family"] = "NanumGothic"   # 그래프 한글 깨짐 방지
plt.rcParams["axes.unicode_minus"] = False

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "GPU 런타임 아님: 런타임 → 런타임 유형 변경 → GPU"
print("device:", device, "| torch", torch.__version__, "| smp", smp.__version__)

device: cuda | torch 2.11.0+cu130 | smp 0.5.0


## 1. 설정

In [3]:
CFG = dict(
    data_dir      = Path(DATA_DIR),   # train/ valid/ test/ 가 있는 폴더
    encoder       = "resnet34",       # 다음 후보: "efficientnet-b3", "resnet50"
    encoder_weights = "imagenet",
    crop          = 512,              # 학습 패치 크기
    crops_per_img = 4,                # 에폭당 이미지 1장에서 뽑는 패치 수
    batch_size    = 8,                # 메모리 부족하면 4로
    epochs        = 80,
    lr            = 3e-4,
    val_every     = 5,                # 몇 에폭마다 검증할지
    seed          = 42,
    ckpt_dir      = Path(DRIVE_DIR) / "checkpoints",   # Drive에 저장 → 세션 끊겨도 유지
    pred_dir      = Path(DRIVE_DIR) / "predictions",
)
CFG["ckpt_dir"].mkdir(parents=True, exist_ok=True)

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
seed_everything(CFG["seed"])

## 2. 데이터 불러오기

In [4]:
CLASSES = ["Al", "Al3Ni", "Primary Si", "Eutectic Si"]
COLORS  = np.array([[153,127,76], [25,76,153], [76,178,76], [204,204,204]], dtype=np.uint8)
N_CLS = len(CLASSES)

def load_gray(path):
    return np.array(Image.open(path).convert("L"))          # (H,W) uint8

def rgb_to_label(path):
    rgb = np.array(Image.open(path).convert("RGB"))
    lab = np.full(rgb.shape[:2], 255, np.uint8)
    for i, c in enumerate(COLORS):
        lab[(rgb == c).all(-1)] = i
    assert (lab != 255).all(), f"정의되지 않은 색이 있음: {path}"
    return lab

def label_to_rgb(lab):
    return COLORS[lab]

def load_split(split):
    imgs = sorted(glob.glob(str(CFG["data_dir"] / split / "images" / "*.png")))
    data = []
    for p in imgs:
        name = Path(p).name.replace("_image.png", "")
        m = CFG["data_dir"] / split / "masks" / f"{name}_mask.png"
        data.append(dict(name=name, img=load_gray(p),
                         mask=rgb_to_label(m) if m.exists() else None))
    return data

train_data = load_split("train")
valid_data = load_split("valid")
test_data  = load_split("test")
print(f"train {len(train_data)} | valid {len(valid_data)} | test {len(test_data)}")

cnt = np.bincount(np.concatenate([d["mask"].ravel() for d in train_data]), minlength=N_CLS)
for c, n in zip(CLASSES, cnt): print(f"  {c:12s} {n/cnt.sum()*100:5.1f}%")

train 70 | valid 20 | test 10
  Al            57.3%
  Al3Ni         26.3%
  Primary Si     9.4%
  Eutectic Si    7.0%


## 3. Dataset과 증강

In [5]:
MEAN, STD = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])

train_tf = A.Compose([
    A.Affine(scale=(0.8, 1.25), rotate=(-15, 15), border_mode=cv2.BORDER_REFLECT_101, p=0.5),
    A.PadIfNeeded(CFG["crop"], CFG["crop"], border_mode=cv2.BORDER_REFLECT_101),
    A.RandomCrop(CFG["crop"], CFG["crop"]),
    A.HorizontalFlip(), A.VerticalFlip(), A.RandomRotate90(),
    A.RandomBrightnessContrast(0.3, 0.3, p=0.8),
    A.RandomGamma((70, 150), p=0.5),
    A.GaussianBlur(blur_limit=(3, 3), p=0.1),
])

def to_tensor(gray):
    x = np.repeat(gray[..., None], 3, -1).astype(np.float32) / 255.0   # 흑백 → 3채널 (ImageNet 가중치 사용)
    x = (x - MEAN) / STD
    return torch.from_numpy(x.transpose(2, 0, 1)).float()

class PatchDataset(torch.utils.data.Dataset):
    def __init__(self, data, tf, n):
        self.data, self.tf, self.n = data, tf, n
    def __len__(self):
        return len(self.data) * self.n
    def __getitem__(self, i):
        d = self.data[i % len(self.data)]
        out = self.tf(image=d["img"], mask=d["mask"])
        return to_tensor(out["image"]), torch.from_numpy(out["mask"]).long()

def make_loader(data):
    ds = PatchDataset(data, train_tf, CFG["crops_per_img"])
    # Colab(리눅스)에서는 워커 2개로 증강을 GPU 학습과 병렬 처리
    return torch.utils.data.DataLoader(ds, batch_size=CFG["batch_size"], shuffle=True,
                                       num_workers=2, pin_memory=True, persistent_workers=True,
                                       drop_last=True)

xb, yb = next(iter(make_loader(train_data)))
print(xb.shape, yb.shape, yb.unique())

torch.Size([8, 3, 512, 512]) torch.Size([8, 512, 512]) tensor([0, 1, 2, 3])


## 4. 모델 · 손실함수

In [6]:
def build_model():
    return smp.Unet(CFG["encoder"], encoder_weights=CFG["encoder_weights"],
                    in_channels=3, classes=N_CLS).to(device)

ce_loss   = nn.CrossEntropyLoss()
dice_loss = smp.losses.DiceLoss(mode="multiclass")
def criterion(logits, y):
    return ce_loss(logits, y) + dice_loss(logits, y)

## 5. 추론 · 평가 함수

In [7]:
@torch.no_grad()
def predict_proba(model, gray, tta=True):
    model.eval()
    H, W = gray.shape
    ph, pw = (-H) % 32, (-W) % 32
    g = np.pad(gray, ((0, ph), (0, pw)), mode="reflect")
    x = to_tensor(g)[None].to(device)
    flips = [[], [3], [2], [2, 3]] if tta else [[]]
    prob = 0
    for f in flips:
        xi = torch.flip(x, f) if f else x
        p = model(xi).softmax(1)
        prob = prob + (torch.flip(p, f) if f else p)
    prob = (prob / len(flips))[0, :, :H, :W]
    return prob.cpu().numpy()                      # (C,H,W)

def iou_per_class(pred, gt):
    ious = []
    for c in range(N_CLS):
        inter = np.logical_and(pred == c, gt == c).sum()
        union = np.logical_or(pred == c, gt == c).sum()
        ious.append(inter / union if union > 0 else np.nan)
    return np.array(ious)

def evaluate(model, data, tta=True):
    per_img, per_cls = [], []
    for d in data:
        pred = predict_proba(model, d["img"], tta).argmax(0)
        ious = iou_per_class(pred, d["mask"])
        per_cls.append(ious); per_img.append(np.nanmean(ious))
    return float(np.mean(per_img)), np.nanmean(np.array(per_cls), 0), per_img

## 6. 학습 함수 (정의만, ResNet34 재학습은 하지 않음)

In [8]:
def train(train_set, val_set=None, epochs=CFG["epochs"], save_name="best.pt"):
    seed_everything(CFG["seed"])
    model = build_model()
    loader = make_loader(train_set)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG["lr"], total_steps=epochs * len(loader),
                                                pct_start=0.05)
    best, hist = -1, []
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time(); tot = 0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            loss = criterion(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            tot += loss.item()
        msg = f"ep {ep:3d} | loss {tot/len(loader):.4f} | {time.time()-t0:.0f}s"
        if val_set is not None and (ep % CFG["val_every"] == 0 or ep == epochs):
            miou, cls, _ = evaluate(model, val_set, tta=False)
            hist.append((ep, miou))
            msg += f" | val mIoU {miou*100:.2f}  " + " ".join(f"{c[:6]}:{v*100:.1f}" for c, v in zip(CLASSES, cls))
            if miou > best:
                best = miou; torch.save(model.state_dict(), CFG["ckpt_dir"] / save_name); msg += "  ★"
        print(msg)
    if val_set is None:
        torch.save(model.state_dict(), CFG["ckpt_dir"] / save_name)
    return model, hist


## 7. 기존 ResNet34 + 8방향 TTA
90도 회전 4가지 × 좌우 뒤집기. 미세조직은 방향성이 없어서 회전 TTA도 유효할 것으로 판단함.

In [9]:
# 8방향 TTA (90도 회전 4가지 x 좌우뒤집기) — 미세조직은 방향성이 없어서 회전도 유효
@torch.no_grad()
def predict_proba8(model, gray):
    model.eval()
    H, W = gray.shape
    ph, pw = (-H) % 32, (-W) % 32
    g = np.pad(gray, ((0, ph), (0, pw)), mode="reflect")
    x = to_tensor(g)[None].to(device)
    prob = 0
    for k in range(4):
        xr = torch.rot90(x, k, (2, 3))
        for f in [[], [3]]:
            xi = torch.flip(xr, f) if f else xr
            p = model(xi).softmax(1)
            p = torch.flip(p, f) if f else p
            prob = prob + torch.rot90(p, -k, (2, 3))
    return (prob / 8)[0, :, :H, :W].cpu().numpy()

def load_model(encoder, fname):
    m = smp.Unet(encoder, encoder_weights=None, in_channels=3, classes=N_CLS).to(device)
    m.load_state_dict(torch.load(CFG["ckpt_dir"] / fname, map_location=device))
    return m.eval()

def evaluate_fn(prob_fn, data):
    per_img, per_cls = [], []
    for d in data:
        ious = iou_per_class(prob_fn(d["img"]).argmax(0), d["mask"])
        per_cls.append(ious); per_img.append(np.nanmean(ious))
    return float(np.mean(per_img)), np.nanmean(np.array(per_cls), 0), per_img

results = {}
def report(name, prob_fn):
    miou, cls, _ = evaluate_fn(prob_fn, valid_data)
    results[name] = miou
    print(f"{name:26s} mIoU {miou*100:.2f} | " + " | ".join(f"{c} {v*100:.1f}" for c, v in zip(CLASSES, cls)))

m_r34 = load_model("resnet34", "best.pt")
report("R34 TTA4 (기존 제출)", lambda g: predict_proba(m_r34, g))
report("R34 TTA8", lambda g: predict_proba8(m_r34, g))

R34 TTA4 (기존 제출)           mIoU 76.31 | Al 92.3 | Al3Ni 70.6 | Primary Si 89.0 | Eutectic Si 54.9
R34 TTA8                   mIoU 76.41 | Al 92.4 | Al3Ni 70.8 | Primary Si 89.0 | Eutectic Si 55.0


## 8. 두 번째 모델: EfficientNet-B3 학습 (T4 약 30분)
베이스라인에서 50에폭 이후 정체돼서 60에폭으로 줄임. 최고 가중치는 `checkpoints/best_effb3.pt`로 저장.

In [10]:
# 두 번째 모델: EfficientNet-B3 인코더, train 70장으로만 학습 (T4에서 약 30분)
# 기존 실험에서 50에폭 이후 정체 → 60에폭으로 단축
CFG["encoder"] = "efficientnet-b3"
model_b3, hist_b3 = train(train_data, valid_data, epochs=60, save_name="best_effb3.pt")
CFG["encoder"] = "resnet34"   # 원래대로

config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 49.3MB            

model.safetensors: downloading bytes:           |  0.00B            

ep   1 | loss 2.2736 | 26s
ep   2 | loss 1.7708 | 26s
ep   3 | loss 1.1560 | 26s
ep   4 | loss 0.8719 | 26s
ep   5 | loss 0.6974 | 26s | val mIoU 65.32  Al:87.7 Al3Ni:57.5 Primar:76.3 Eutect:39.8  ★
ep   6 | loss 0.6581 | 26s
ep   7 | loss 0.5860 | 26s
ep   8 | loss 0.5663 | 26s
ep   9 | loss 0.5202 | 26s
ep  10 | loss 0.5141 | 26s | val mIoU 70.55  Al:89.8 Al3Ni:65.2 Primar:81.4 Eutect:45.9  ★
ep  11 | loss 0.4943 | 26s
ep  12 | loss 0.4711 | 26s
ep  13 | loss 0.4785 | 26s
ep  14 | loss 0.4552 | 26s
ep  15 | loss 0.4633 | 26s | val mIoU 72.17  Al:89.9 Al3Ni:66.1 Primar:82.4 Eutect:50.2  ★
ep  16 | loss 0.4609 | 26s
ep  17 | loss 0.4516 | 26s
ep  18 | loss 0.4501 | 26s
ep  19 | loss 0.4499 | 26s
ep  20 | loss 0.4393 | 26s | val mIoU 73.56  Al:91.3 Al3Ni:67.8 Primar:83.4 Eutect:51.7  ★
ep  21 | loss 0.4200 | 26s
ep  22 | loss 0.4222 | 26s
ep  23 | loss 0.4285 | 26s
ep  24 | loss 0.4198 | 26s
ep  25 | loss 0.4159 | 26s | val mIoU 73.52  Al:91.6 Al3Ni:66.8 Primar:83.6 Eutect:52.1
ep  26 |

## 9. 단독 · 앙상블 비교 (valid 20장)

In [13]:
# 단독 / 앙상블 비교 (모두 valid 20장, 이미지별 mIoU 평균)
m_b3 = load_model("efficientnet-b3", "best_effb3.pt")
CANDIDATES = {
    "R34 TTA4 (기존 제출)": lambda g: predict_proba(m_r34, g),
    "R34 TTA8":            lambda g: predict_proba8(m_r34, g),
    "B3 TTA8":             lambda g: predict_proba8(m_b3, g),
    "앙상블 R34+B3 TTA4":  lambda g: (predict_proba(m_r34, g) + predict_proba(m_b3, g)) / 2,
    "앙상블 R34+B3 TTA8":  lambda g: (predict_proba8(m_r34, g) + predict_proba8(m_b3, g)) / 2,
}
for name, fn in CANDIDATES.items():
    if name not in results: report(name, fn)

BEST = max(results, key=results.get)
print(f"\n▶ 최고: {BEST}  ({results[BEST]*100:.2f}%)")


▶ 최고: 앙상블 R34+B3 TTA8  (76.76%)


## 10. 최고 조합으로 test 예측 저장
**기존 제출(R34 TTA4)이 최고로 나오면 이 결과는 쓰지 않고** 기존 `predictions/`를 제출함.

In [14]:
# 최고 조합으로 test 예측 저장 → predictions_v2/ (기존 predictions/는 그대로)
# 기존 제출(R34 TTA4)이 최고라면 이 셀은 실행할 필요 없음
pred_v2 = Path(DRIVE_DIR) / "predictions_v2"
pred_v2.mkdir(exist_ok=True)
fn = CANDIDATES[BEST]
for d in test_data:
    pred = fn(d["img"]).argmax(0).astype(np.uint8)
    rgba = np.concatenate([label_to_rgb(pred), np.full(pred.shape + (1,), 255, np.uint8)], -1)
    Image.fromarray(rgba).save(pred_v2 / f"{d['name']}_mask.png")
print(f"{BEST} → {len(test_data)}장 저장:", pred_v2)

import shutil
from google.colab import files
shutil.make_archive("/content/predictions_v2", "zip", pred_v2)
files.download("/content/predictions_v2.zip")

앙상블 R34+B3 TTA8 → 10장 저장: /content/drive/MyDrive/KoMaP/predictions_v2


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>